# WaveSpeedAI Jupyter Client (RunPod)
Run the cells in order.

In [ ]:
# Cell 1 - Install dependencies
%pip install httpx==0.27.0

In [ ]:
# Cell 2 - API key setup and validation
import httpx

WAVESPEED_API_KEY = input("Enter your WaveSpeedAI API key: ").strip()
BASE_URL = "https://api.wavespeed.ai/v3"

if not WAVESPEED_API_KEY:
    raise ValueError("WAVESPEED_API_KEY is empty. Paste your key in this cell.")

def _auth_headers():
    return {"Authorization": f"Bearer {WAVESPEED_API_KEY}"}

with httpx.Client(base_url=BASE_URL, headers=_auth_headers(), timeout=30) as client:
    response = client.get("/models")
    if response.status_code in (401, 403):
        raise PermissionError(f"Auth failed ({response.status_code}): {response.text}")
    response.raise_for_status()
    print("API key validated. Model count:", len(response.json().get("data", [])))


In [ ]:
# Cell 3 - WaveSpeedClient (httpx sync)
from typing import Any, Dict, List, Optional
import time

class WaveSpeedClient:
    def __init__(self, api_key: str, base_url: str = "https://api.wavespeed.ai/v3") -> None:
        self.api_key = api_key
        self.base_url = base_url
        self.client = httpx.Client(
            base_url=self.base_url,
            headers={"Authorization": f"Bearer {self.api_key}"},
            timeout=120,
        )

    def _handle_response(self, response: httpx.Response) -> Dict[str, Any]:
        if response.status_code in (401, 403):
            raise PermissionError(f"Auth failed ({response.status_code}): {response.text}")
        response.raise_for_status()
        return response.json()

    def upload_file(self, local_path: str) -> str:
        print(f"Uploading file: {local_path}")
        with open(local_path, "rb") as handle:
            files = {"file": handle}
            response = self.client.post("/files", files=files)
        payload = self._handle_response(response)
        file_id = payload.get("id") or payload.get("file_id")
        if not file_id:
            raise RuntimeError(f"Upload response missing file id: {payload}")
        print(f"Uploaded file id: {file_id}")
        return file_id

    def list_models(self) -> List[Dict[str, Any]]:
        response = self.client.get("/models")
        payload = self._handle_response(response)
        return payload.get("data", payload)

    def run_model(self, model_id: str, payload: Dict[str, Any]) -> Dict[str, Any]:
        print(f"Submitting job to model: {model_id}")
        response = self.client.post(f"/generate/{model_id}", json=payload)
        return self._handle_response(response)

    def poll_job(self, job_id: str) -> Dict[str, Any]:
        response = self.client.get(f"/jobs/{job_id}")
        return self._handle_response(response)

    def wait_for_completion(self, job_id: str, poll_interval: float = 3.0, timeout: float = 900) -> Dict[str, Any]:
        print(f"Polling job: {job_id}")
        start = time.time()
        while True:
            payload = self.poll_job(job_id)
            status = payload.get("status")
            print(f"Status: {status}")
            if status in {"succeeded", "failed", "canceled"}:
                return payload
            if time.time() - start > timeout:
                raise TimeoutError("Timed out waiting for job completion")
            time.sleep(poll_interval)

    def download_file(self, url: str) -> bytes:
        print(f"Downloading output: {url}")
        response = self.client.get(url)
        if response.status_code in (401, 403):
            raise PermissionError(f"Auth failed ({response.status_code}): {response.text}")
        response.raise_for_status()
        return response.content

client = WaveSpeedClient(WAVESPEED_API_KEY, BASE_URL)


In [ ]:
# Cell 4 - List models
models = client.list_models()
print(f"Loaded {len(models)} models")
for model in models[:10]:
    print(f"- {model.get('id')} | {model.get('name')}")


In [ ]:
# Cell 5 - Local image -> image-to-video example
from pathlib import Path

local_image_path = input("Path to local image file: ").strip()
if not Path(local_image_path).exists():
    raise FileNotFoundError(f"Image not found: {local_image_path}")

image_file_id = client.upload_file(local_image_path)

image_video_models = [
    m for m in models
    if "id" in m and (m.get("capabilities", {}).get("video") or "video" in m.get("input_schema", {}).get("properties", {}))
]
if not image_video_models:
    raise RuntimeError("No image-to-video capable models found in the model list.")

model_id = image_video_models[0]["id"]
print(f"Using model: {model_id}")

payload = {
    "prompt": "A cinematic flythrough of a futuristic city at sunset",
    "image": image_file_id,
    "duration": 4,
    "steps": 30,
    "guidance_scale": 7.5,
}

job_response = client.run_model(model_id, payload)
job_id = job_response.get("job_id") or job_response.get("id")
if not job_id:
    raise RuntimeError(f"Job submission failed: {job_response}")

result = client.wait_for_completion(job_id)
if result.get("status") != "succeeded":
    raise RuntimeError(f"Job failed: {result}")

output_urls = result.get("outputs") or result.get("output_urls") or []
print("Outputs:", output_urls)


In [ ]:
# Cell 6 - Save outputs to /workspace/outputs
from pathlib import Path

output_dir = Path("/workspace/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

saved_paths = []
for index, output in enumerate(output_urls, start=1):
    url = output.get("url") if isinstance(output, dict) else output
    if not url:
        continue
    content = client.download_file(url)
    suffix = ".bin"
    if isinstance(url, str) and "." in url:
        suffix = "." + url.split(".")[-1].split("?")[0]
    path = output_dir / f"wavespeed_output_{index}{suffix}"
    path.write_bytes(content)
    saved_paths.append(str(path))
    print(f"Saved: {path}")

saved_paths
